# Job Matching Engine — Data Exploration & TF-IDF Baseline

Pipeline: Kaggle dataset + manually collected German job postings -> cleaning -> bilingual TF-IDF matching against my CV.

In [1]:
import os

# Ensure working directory is the project root, not notebooks/
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print("Working directory:", os.getcwd())

Working directory: C:\Users\Administrator\IdeaProjects\job-matching-engine


## 1. Load Kaggle dataset

In [2]:
import os
import pandas as pd

if not os.path.exists('data/raw/job_postings.csv'):
    import kaggle
    kaggle.api.dataset_download_files(
        'asaniczka/data-science-job-postings-and-skills',
        path='data/raw', unzip=True
    )

postings = pd.read_csv('data/raw/job_postings.csv')
skills = pd.read_csv('data/raw/job_skills.csv')
summary = pd.read_csv('data/raw/job_summary.csv')

print(postings.shape, skills.shape, summary.shape)

(12217, 15) (12217, 2) (12217, 2)


In [3]:
df = postings.merge(skills, on='job_link', how='left') \
    .merge(summary, on='job_link', how='left')

print(df.shape)
print(df.columns.tolist())
df.head(2)

(12217, 17)
['job_link', 'last_processed_time', 'last_status', 'got_summary', 'got_ner', 'is_being_worked', 'job_title', 'company', 'job_location', 'first_seen', 'search_city', 'search_country', 'search_position', 'job_level', 'job_type', 'job_skills', 'job_summary']


,job_link,last_processed_time,last_status,got_summary,got_ner,is_being_worked,job_title,company,job_location,first_seen,search_city,search_country,search_position,job_level,job_type,job_skills,job_summary
0,https://www.linkedin.com/jobs/view/senior-mach...,2024-01-21 08:08:48.031964+00,Finished NER,t,t,f,Senior Machine Learning Engineer,Jobs for Humanity,"New Haven, CT",2024-01-14,East Haven,United States,Agricultural-Research Engineer,Mid senior,Onsite,"Machine Learning, Programming, Python, Scala, ...",Company Description\nJobs for Humanity is part...
1,https://www.linkedin.com/jobs/view/principal-s...,2024-01-20 04:02:12.331406+00,Finished NER,t,t,f,"Principal Software Engineer, ML Accelerators",Aurora,"San Francisco, CA",2024-01-14,El Cerrito,United States,Set-Key Driver,Mid senior,Onsite,"C++, Python, PyTorch, TensorFlow, MXNet, CUDA,...",Who We Are\nAurora (Nasdaq: AUR) is delivering...


## 2. Explore the data

In [4]:
# Check missing values per column
print(df.isnull().sum())

# Check distribution of job titles
print(df['job_title'].value_counts().head(20))

# Preview raw job summary text
print(df.loc[0, 'job_summary'])

job_link               0
last_processed_time    0
last_status            0
got_summary            0
got_ner                0
is_being_worked        0
job_title              0
company                0
job_location           1
first_seen             0
search_city            0
search_country         0
search_position        0
job_level              0
job_type               0
job_skills             5
job_summary            0
dtype: int64
job_title
Senior Data Engineer                                        285
Senior Data Analyst                                         163
Data Engineer                                               149
Senior MLOps Engineer                                       138
Data Analyst                                                137
Data Scientist                                              128
Lead Data Engineer                                          123
Senior Data Scientist                                       119
Data Architect                          

## 3. Clean the data

Drop missing critical fields and exact-duplicate postings.

In [5]:
# Drop rows with missing critical fields
df = df.dropna(subset=['job_location', 'job_skills']).reset_index(drop=True)

# Check for duplicate postings
print(f"Duplicate job_links: {df['job_link'].duplicated().sum()}")
df = df.drop_duplicates(subset=['job_link']).reset_index(drop=True)

print(df.shape)

Duplicate job_links: 0
(12211, 17)


### Text cleaning function

Strips boilerplate (legal/diversity notices, application instructions) from the tail of each posting.
Handles **both English and German** markers, since the final dataset combines Kaggle (EN) postings with manually collected German postings.

Only strips boilerplate found in the last 15% of the text.  An earlier version stripped from the *first* occurrence anywhere in the text, which accidentally wiped out entire postings when a marker like "reasonable accommodations" appeared early in a legitimate description (found via a validation check, see below).

In [6]:
import re

def clean_job_summary(text):
    if not isinstance(text, str):
        return ""

    # Only strip boilerplate if it appears in the last 15% of the text
    # Bilingual boilerplate markers (English + German)
    boilerplate_markers = [
        "show more", "equal opportunity employer",
        "protected veteran status", "reasonable accommodations",
        "chancengleichheit", "wir freuen uns auf deine bewerbung",
        "bewerbungsunterlagen", "vielfalt und inklusion",
    ]

    cutoff_point = int(len(text) * 0.85)
    tail = text[cutoff_point:].lower()

    for marker in boilerplate_markers:
        idx = text.lower().find(marker, cutoff_point)
        if idx != -1:
            text = text[:idx]
            break

    text = re.sub(r"\s+", " ", text).strip()
    return text

df['job_summary_clean'] = df['job_summary'].apply(clean_job_summary)

print(df['job_summary_clean'].str.len().describe())

count    12211.000000
mean      4256.025469
std       2293.421298
min         21.000000
25%       2541.500000
50%       3972.000000
75%       5708.000000
max      19177.000000
Name: job_summary_clean, dtype: float64


In [7]:
# Validation: check no posting got wiped to empty by cleaning
empty_after_clean = df[df['job_summary_clean'].str.len() == 0]
print(f"Number of rows now empty: {len(empty_after_clean)}")

Number of rows now empty: 0


Remove non-relevant administrative/procedural postings and reposted duplicates (same title + company, different link — common with recruiting agencies reposting the same listing).

In [8]:
# Flag postings that are mostly procedural/administrative
admin_keywords = ['CalCareer', 'Examination/Employment Application', 'Statement of Qualifications']
df['is_admin_posting'] = df['job_summary'].str.contains('|'.join(admin_keywords), case=False, na=False)

print(f"Admin/procedural postings detected: {df['is_admin_posting'].sum()}")

df_filtered = df[~df['is_admin_posting']].reset_index(drop=True)

# Remove reposted duplicates (same title + company, different job_link)
before = df_filtered.shape[0]
df_filtered = df_filtered.drop_duplicates(subset=['job_title', 'company']).reset_index(drop=True)
after = df_filtered.shape[0]
print(f"Removed {before - after} duplicate postings (same title + company)")

print(df_filtered.shape)

Admin/procedural postings detected: 24
Removed 3363 duplicate postings (same title + company)
(8824, 19)


## 4. Load my CV (reference text for matching)

In [9]:
with open('data/my_cv.txt', 'r', encoding='utf-8') as f:
    my_cv = f.read()

print(f"CV length: {len(my_cv)} characters")
print(my_cv[:300])

CV length: 4692 characters
﻿Pharel Harold Nanseu Kombou

Data Science, Machine Learning & MLOps – Werkstudent / Praktikant

Gießen, Deutschland | 0162571365 | haroldpharel@gmail.com | linkedin.com/in/pharel-nanseu-042281356 | github.com/Pharel8

PROFIL

Informatik-Student (B.Sc., THM Gießen, 5. Semester) mit Schwerpunkt Data 


## 5. Load manually collected German job postings

10 real Werkstudent/Praktikum postings (LinkedIn/StepStone/Xing) collected to complement the Kaggle dataset, which is US/English-only.

In [10]:
manual_offers = pd.read_csv('data/manual_offers.csv')
manual_offers.columns = manual_offers.columns.str.strip()
manual_offers = manual_offers.dropna(subset=['job_title', 'company']).reset_index(drop=True)
manual_offers['company'] = manual_offers['company'].str.strip()
manual_offers['job_summary_clean'] = manual_offers['job_summary'].apply(clean_job_summary)

print(manual_offers.shape)
print(manual_offers['job_summary_clean'].str.len().describe())

(10, 7)
count      10.000000
mean     1137.300000
std       347.241847
min       804.000000
25%       964.750000
50%      1083.000000
75%      1149.250000
max      2027.000000
Name: job_summary_clean, dtype: float64


## 6. Combine Kaggle + manual offers into one dataset

In [11]:
combined = pd.concat([
    df_filtered[['job_title', 'company', 'job_location', 'job_summary_clean', 'job_skills']],
    manual_offers[['job_title', 'company', 'job_location', 'job_summary_clean', 'job_skills']]
], ignore_index=True)

print(f"Kaggle offers: {len(df_filtered)}")
print(f"Manual offers: {len(manual_offers)}")
print(f"Combined total: {len(combined)}")

Kaggle offers: 8824
Manual offers: 10
Combined total: 8834


## 7. TF-IDF matching — bilingual stopwords

**Finding during development:** using only English stopwords (`stop_words='english'`) with a German CV against this bilingual corpus caused generic German HR vocabulary ("und", "für", "mit"...) to dominate the similarity score. This produced a very high top score (~0.82) but pulled in false positives — e.g. "Teamleiter Produktion" and "Projektleiter Großschaden" ranked highly purely because they were in German, not because they were relevant to a Data Science profile.

Adding German stopwords alongside English ones fixed this: scores dropped to a more realistic ~0.44, and the false positives disappeared — the top results became consistently ML/Data Science roles.

In [12]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words('english'))
german_stopwords = set(stopwords.words('german'))
combined_stopwords = list(english_stopwords | german_stopwords)

print(f"Total combined stopwords: {len(combined_stopwords)}")

Total combined stopwords: 424


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Administrator\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine CV with all job summaries into one corpus
corpus = [my_cv] + combined['job_summary_clean'].tolist()

vectorizer = TfidfVectorizer(stop_words=combined_stopwords, max_features=5000)
tfidf_matrix = vectorizer.fit_transform(corpus)

# CV is the first row (index 0), compare it against all job postings (index 1 onward)
cv_vector = tfidf_matrix[0:1]
job_vectors = tfidf_matrix[1:]

similarity_scores = cosine_similarity(cv_vector, job_vectors).flatten()

combined['tfidf_score'] = similarity_scores

# Show top 15 matches
top_matches = combined.sort_values('tfidf_score', ascending=False).head(15)
print(top_matches[['job_title', 'company', 'tfidf_score']])

                                              job_title  \
8825           Werkstudent (m/w/d) Data Science - Hotel   
8833       Werkstudent Data Analytics / Computer Vision   
8832        Werkstudent Data Engineering & Data Science   
8831  WerkstudentIn Data Science, Machine Learning & AI   
5811                          Machine Learning Engineer   
6037                 Machine Learning Software Engineer   
8569                              Senior Data Scientist   
1105                              Senior MLOps Engineer   
3873             Machine Learning Platform Engineer x 2   
421               Senior Machine Learning Engineer - AI   
3725                          Machine Learning Engineer   
3906                 Machine Learning Platform Engineer   
5513                   Senior Machine Learning Engineer   
8826  Werkstudent Data Science & Machine Learning (m...   
6180            ML Engineer - Data & Advanced Analytics   

                                                company